# Notebook 4: Consolidates Emissions Data (ETS)

Comlpiance datasets contains installation-year level data; consisting of emissions-related and compliance information such as yearly verified emissions, surrendered carbon allowance, free carbon allocations, per installation-year.

*objective*: Aggregate related information from installation-year to parent company-year grain of information:
The company-year output of Notebook 04 should carry, per (company_id, year):
- ets_verified = Σ verified (the emissions signal)
- ets_allocated_free = Σ allocatedFree (the subsidy signal)
- and derived-later: net position = allocated − verified

summed across installations belonging to that company, within the kept systems only.


In [1]:
# %% make repo root importable (so `from src...` works)
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

root on path: /Users/admin/Desktop/carbon-portfolio-project-v2


In [2]:
# %% auto-reload edited src modules (so src/*.py edits take effect without kernel restart)
%load_ext autoreload
%autoreload 2

In [3]:
import sqlite3, pandas as pd
from src.db import connect, load_table
from src.config import DATA_PROCESSED

In [5]:
compliance = pd.read_csv('/Users/admin/Desktop/carbon-portfolio-project-v2/data/raw/compliance.csv')

/var/folders/bg/1y0v_kw15cnb9w8c866jb_280000gn/T/ipykernel_97765/3727503339.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  compliance = pd.read_csv('/Users/admin/Desktop/carbon-portfolio-project-v2/data/raw/compliance.csv')


In [13]:
con = connect(str(ROOT / 'data' / 'carbon.db'))
DB = '/Users/admin/Desktop/carbon-portfolio-project-v2/data/carbon.db'
con.executescript((ROOT / 'sql' / 'schema.sql').read_text())


### Profiling compliance data

Three reporting systems under ETS:

- euets (EU ETS proper). This is your core signal, and the ETS-comparable-to-Trucost-Scope-1 number.
- chets (the Swiss ETS). Linked to EU ETS since 2020
- esd (Effort Sharing Decision): covers non-ETS sectors (transport, buildings, agriculture, small industry) at the member-state level, not the installation level. ESD "emissions" are not the same quantity as ETS verified emissions. Mixing ESD into a company's ETS emissions would conflate two incompatible accounting systems. Thus, they are excluded.

In [16]:
# 1. rows and verified-emissions mass by system
print(compliance.groupby('reportedInSystem_id').agg(
    rows=('verified', 'size'), # size is count
    has_verified=('verified', lambda s: s.notna().sum()), #
    not_verfied=('verified',lambda s: s.isna().sum()),
    total_verified=('verified', 'sum')).to_string())


                       rows  has_verified  not_verfied  total_verified
reportedInSystem_id                                                   
chets                 15290          1366        13924    1.790420e+06
esd                     224           224            0    2.015527e+10
euets                509496        207773       301723    3.393607e+10


In [18]:
# 2. THE key check: does any installation-year appear under >1 system? (double-count risk)
dup = (compliance.groupby(['installation_id', 'year'])['reportedInSystem_id'].nunique())
print(f"\ninstallation-years in >1 system: {(dup > 1).sum():,}")



installation-years in >1 system: 15,290


In [23]:

# 3. do esd rows even carry verified emissions, or are they blank?
print("\nesd verified non-null:",
      compliance.loc[compliance['reportedInSystem_id']=='esd', 'verified'].notna().sum())


esd verified non-null: 224


### Section A: Linking Installations to Company ID


In [41]:
ec = pd.read_sql('SELECT bvd_id_number, company_id FROM ets_entity_company', con)
g1 = ec['bvd_id_number'].duplicated().sum()
print(f"GUARD 1  bvd → multiple company_id : {g1}   (expect 0)")


GUARD 1  bvd → multiple company_id : 0   (expect 0)


In [43]:
full = pd.read_csv(DATA_PROCESSED / 'full_ets_samples.csv', low_memory=False)

# instllation, and bvdid map:
imap = (full.loc[full['bvdId'].notna(), ['installation_id', 'bvdId', 'operator_type']]
            .rename(columns={'bvdId': 'bvd_id_number'})
            .drop_duplicates())

In [47]:
# GUARD 2: one installation → one BvD (else compliance rows fan out) -- this means that the installations may have changed accounts
g2 = (imap.groupby('installation_id')['bvd_id_number'].nunique() > 1).sum()
print(f"GUARD 2  installation → multiple BvD : {g2}   (expect 0)")

GUARD 2  installation → multiple BvD : 10   (expect 0)


In [46]:
imap = imap.merge(ec, on='bvd_id_number', how='inner')
print(f"\ninstallations w/ listed parent : {imap['installation_id'].nunique():,}")
print(f"companies covered              : {imap['company_id'].nunique():,}")
print(imap['operator_type'].value_counts().to_string())


installations w/ listed parent : 2,629
companies covered              : 300
operator_type
stationary    2600
aircraft        39


In [48]:
# how many multi-BvD installations resolve to >1 company_id (the real ambiguity)?


per_inst = imap.groupby('installation_id')['company_id'].nunique()
split = per_inst[per_inst > 1]
print(f"installations → multiple company_id : {len(split)}   (these need a rule)")
print(f"installations → single company_id   : {(per_inst==1).sum()}   (safe, dedup)")

installations → multiple company_id : 5   (these need a rule)
installations → single company_id   : 2624   (safe, dedup)


In [50]:
bad = split.index                      # installation_ids with >1 company_id
cols = ['installation_id','bvdId','operator_type','openingDate','closingDate']
insp = (full[full['installation_id'].isin(bad)][cols]
        .merge(ec.rename(columns={'bvd_id_number':'bvdId'}), on='bvdId', how='left')
        .sort_values(['installation_id','openingDate']))
print(insp.to_string(index=False))

installation_id         bvdId operator_type         openingDate         closingDate   company_id
          ES_45   ESB39868823    stationary 2005-02-28 00:00:00 2022-12-12 12:52:19 PTEDP0AM0009
          ES_45   ESB62733126    stationary                 NaN 2022-12-12 12:51:35 DE000ENAG999
         ES_746   ESA95000295    stationary 2005-02-28 00:00:00                 NaN FR0000120271
         ES_746   ESA33473752    stationary                 NaN                 NaN PTEDP0AM0009
         HU_154    HU10782004    stationary 2006-04-10 00:00:00                 NaN FR0000124141
         HU_154    HU11153803    stationary                 NaN                 NaN DE000ENAG999
          HU_41    HU10782004    stationary 2006-04-10 00:00:00                 NaN FR0000124141
          HU_41    HU11153803    stationary                 NaN                 NaN DE000ENAG999
        IT_1164 IT01734970120    stationary 2010-10-06 00:00:00                 NaN IT0001233417
        IT_1164 IT95012280137 

In [51]:
full['openingDate'] = pd.to_datetime(full['openingDate'], errors='coerce')

resolved = (full.loc[full['bvdId'].notna(),
                     ['installation_id','bvdId','operator_type','openingDate']]
                .rename(columns={'bvdId':'bvd_id_number'})
                .merge(ec, on='bvd_id_number', how='inner'))

# prefer dated operator; if two dated, latest openingDate wins
resolved['has_open'] = resolved['openingDate'].notna()
imap = (resolved.sort_values(['installation_id','has_open','openingDate'],
                             ascending=[True, False, False])
                .drop_duplicates('installation_id', keep='first')
                .drop(columns=['has_open','openingDate']))

assert imap['installation_id'].is_unique
print(f"installations mapped : {imap['installation_id'].nunique():,}")
print(f"companies covered    : {imap['company_id'].nunique():,}")
print(imap['operator_type'].value_counts().to_string())

installations mapped : 2,629
companies covered    : 300
operator_type
stationary    2590
aircraft        39


### Section B: Reconcile Compliance Features to Company-Year

In [71]:
verified_comp = comp[comp['verified'].notna()]
verified_comp[verified_comp['compliance_id'] != 'A']

,installation_id,year,reportedInSystem_id,euetsPhase,compliance_id,allocatedFree,allocatedNewEntrance,allocatedTotal,allocated10c,verified,verifiedCummulative,verifiedUpdated,surrendered,surrenderedCummulative,balance,penalty,created_on,updated_on
131,AT_200109,2010,euets,2008-2012,NaN,NaN,NaN,0,NaN,0.0,NaN,False,NaN,NaN,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592
132,AT_200109,2011,euets,2008-2012,NaN,NaN,NaN,0,NaN,0.0,NaN,False,NaN,NaN,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592
490,AT_62,2005,euets,2005-2007,B,8495.0,NaN,8495,NaN,9754.0,9754.0,False,NaN,0.0,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592
1117,AT_203,2008,euets,2008-2012,E,135474.0,NaN,135474,NaN,135519.0,135519.0,True,135519.0,60693.0,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592
1118,AT_203,2009,euets,2008-2012,E,135474.0,NaN,135474,NaN,131537.0,267056.0,True,131537.0,334040.0,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523905,AT_37,2009,euets,2008-2012,B,9981.0,NaN,9981,NaN,3495.0,10902.0,False,3495.0,5865.0,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592
523906,AT_37,2010,euets,2008-2012,B,9981.0,NaN,9981,NaN,3896.0,14798.0,False,3896.0,9761.0,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592
523907,AT_37,2011,euets,2008-2012,B,9981.0,NaN,9981,NaN,919.0,15717.0,False,919.0,10680.0,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592
524644,AT_201018,2015,euets,2013-2020,C,7802.0,NaN,7802,NaN,15330.0,NaN,True,NaN,30127.0,NaN,NaN,2024-11-06 10:00:50.904092,2024-11-06 10:00:50.905592


In [73]:
# ── NB04 · Section B: reconcile compliance → company-year ────────────────────
comp = compliance[compliance['reportedInSystem_id'].isin(['euets', 'chets'])].copy()   # drop esd


# find the total verified emissions, allocated free and number in systems per installation-year: sum euets + chets (dual obligation, not double-count):
iy = (comp.groupby(['installation_id', 'year'], as_index=False)
          .agg(verified       = ('verified',      lambda s: s.sum(min_count=1)),
               allocated_free = ('allocatedFree', lambda s: s.sum(min_count=1)),
               surrendered    = ('surrendered',   lambda s: s.sum(min_count=1)),
               n_sys          = ('reportedInSystem_id', 'nunique')))

# coverage funnel
iy = iy.merge(imap, on='installation_id', how='left') # linking company_id to each installation_id
tot_v = iy['verified'].sum()
mat_v = iy.loc[iy['company_id'].notna(), 'verified'].sum()
print(f"verified tonnes — all euets+chets : {tot_v:,.0f}")
print(f"verified tonnes — listed parent   : {mat_v:,.0f}  ({mat_v/tot_v*100:.1f}%)")
print(f"installation-years dropped (no parent) : {iy['company_id'].isna().sum():,}")

# find the total verified emissions, allocated free, n_installations, n_verified_report, and aircraft for company-year:
iy = iy[iy['company_id'].notna()] # filter out installations without any parent company

ets_company_emissions = (iy.groupby(['company_id', 'year'], as_index=False)
    .agg(verified          = ('verified',       lambda s: s.sum(min_count=1)),
         allocated_free    = ('allocated_free', lambda s: s.sum(min_count=1)),
         surrendered       = ('surrendered',    lambda s: s.sum(min_count=1)),
         
         n_installations   = ('installation_id', 'nunique'),
         n_verified_report = ('verified', lambda s: s.notna().sum()),
         has_aircraft      = ('operator_type', lambda s: (s == 'aircraft').any())))

print(f"\nshape : {ets_company_emissions.shape}")
print(f"years : {ets_company_emissions['year'].min()}–{ets_company_emissions['year'].max()}")
print(ets_company_emissions.groupby('year')['verified'].agg(['count', 'sum']).to_string())

verified tonnes — all euets+chets : 33,937,855,923
verified tonnes — listed parent   : 12,525,370,576  (36.9%)
installation-years dropped (no parent) : 441,142

shape : (7800, 8)
years : 2005–2030
      count          sum
year                    
2005    229  696681973.0
2006    236  717747237.0
2007    249  731137322.0
2008    255  722274545.0
2009    256  648567270.0
2010    257  670740808.0
2011    259  674549694.0
2012    271  681375460.0
2013    281  712561872.0
2014    284  686522714.0
2015    282  705219869.0
2016    285  696915983.0
2017    285  708159774.0
2018    286  684037751.0
2019    281  630801739.0
2020    279  554021871.0
2021    251  562909003.0
2022    246  556812436.0
2023    242  484333255.0
2024      0          0.0
2025      0          0.0
2026      0          0.0
2027      0          0.0
2028      0          0.0
2029      0          0.0
2030      0          0.0


In [74]:
# keep FY2012+ (earliest FY knowable within 2013 burn-in via 1-Jul-Y+1 lag)
ets_company_emissions = ets_company_emissions[ets_company_emissions['year'] >= 2012].copy()

print(f"\nshape : {ets_company_emissions.shape}")
print(f"years : {ets_company_emissions['year'].min()}–{ets_company_emissions['year'].max()}")
print(f"post-2023 verified non-null (expect 0) : "
      f"{ets_company_emissions.loc[ets_company_emissions.year>=2024,'verified'].notna().sum()}")


shape : (5700, 8)
years : 2012–2030
post-2023 verified non-null (expect 0) : 0


### Section C: Load Emissions Data into Database

In [80]:
EXTRAS_DDL = """
CREATE TABLE IF NOT EXISTS ets_company_extras (
    company_id TEXT NOT NULL,
    year INTEGER NOT NULL,
    allocated_free REAL,
    surrendered REAL,
    n_installations INTEGER,
    n_verified_report INTEGER,
    has_aircraft INTEGER,
    PRIMARY KEY (company_id, year),
    FOREIGN KEY (company_id)
    REFERENCES master_company_list(company_id)
);
"""

In [82]:
# separate emissions only columns from ets_company_emissions
ce = (ets_company_emissions.loc[ets_company_emissions['verified'].notna(),
                                ['company_id', 'year', 'verified']]
          .rename(columns={'verified': 'scope1_emissions'})
          .assign(source='ets_registry')) # create common column

# create extra ETS-only emissions features (allocated, surrendered) table:
extras = ets_company_emissions[['company_id', 'year', 'allocated_free', 'surrendered',
                                'n_installations', 'n_verified_report', 'has_aircraft']].copy()
extras['has_aircraft'] = extras['has_aircraft'].astype(int)

In [84]:
# Drop and Create extra table:
con.execute("DROP TABLE IF EXISTS emissions;")               # empty, dead FK
con.execute("DROP TABLE IF EXISTS ets_company_emissions;")   # remove earlier parallel table if it was created
con.execute(EXTRAS_DDL) # create new ETS extra table
con.commit()

In [85]:
# Load/append ETS data into existing company_emissions table:
load_table(con, ce, 'company_emissions', )

  company_emissions        3,273 rows


In [86]:
load_table(con, extras, 'ets_company_extras')

  ets_company_extras       5,700 rows


In [87]:
n_ce  = con.execute("SELECT COUNT(*) FROM company_emissions WHERE source='ets_registry'").fetchone()[0]
n_ex  = con.execute("SELECT COUNT(*) FROM ets_company_extras").fetchone()[0]

orphan = con.execute("""
    SELECT COUNT(*) FROM company_emissions e
    LEFT JOIN master_company_list m ON m.company_id = e.company_id
    WHERE m.company_id IS NULL
""").fetchone()[0]

print(f"company_emissions (ets_registry) : {n_ce:,}   (expect 3,600)")
print(f"ets_company_extras               : {n_ex:,}   (expect 5,700)")
print(f"orphans                          : {orphan}   (expect 0)")

company_emissions (ets_registry) : 3,273   (expect 3,600)
ets_company_extras               : 5,700   (expect 5,700)
orphans                          : 0   (expect 0)


In [88]:
m = ets_company_emissions['year'].between(2012, 2023)
print(ets_company_emissions.loc[m, 'verified'].isna().sum())   # expect 327

327


In [90]:
# Some checks: how many company-year appear in more than one row:
print(pd.read_sql("""
    SELECT company_id, year, COUNT(*) n
    FROM company_emissions
    GROUP BY company_id, year
    HAVING n > 1""", con))      


Empty DataFrame
Columns: [company_id, year, n]
Index: []


In [20]:
print(pd.read_sql("""SELECT *
                  FROM company_emissions
                  WHERE source = 'trucost'
                  LIMIT 10
                  """, con))

     company_id  year  scope1_emissions   source
0  DK0060030286  2019            57.557  trucost
1  DK0060030286  2020            79.877  trucost
2  DK0060030286  2021           104.276  trucost
3  DK0060030286  2022            15.246  trucost
4  DK0060030286  2023            20.782  trucost
5  FR0010417345  2020           157.273  trucost
6  FR0010417345  2021            74.651  trucost
7  FR0010417345  2022            58.921  trucost
8  FR0010417345  2023           192.231  trucost
9  FR0010417345  2024            50.457  trucost
